# Feature Engineering — Used Car Price Prediction

Loads the cleaned output from `02_data_cleaning.ipynb` and adds derived
features (`Brand`, `Model`, `Car_Age`), as functions from `src/features.py`.
Runs on the whole dataset — everything here is deterministic string parsing
and arithmetic. Grouping rare `Model` categories, encoding, and imputation
are NOT done here; they depend on training-set statistics and belong in the
preprocessing Pipeline, built after the train/test split.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.features import extract_brand_and_model, add_car_age, engineer_features

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv("../data/processed/used_cars_cleaned.csv")
df.shape

(6018, 14)

In [3]:
df = extract_brand_and_model(df)
df[['Name', 'Brand', 'Model']].head()

,Name,Brand,Model
0,Maruti Wagon R LXI CNG,Maruti,Wagon R LXI CNG
1,Hyundai Creta 1.6 CRDi SX Option,Hyundai,Creta 1.6 CRDi SX Option
2,Honda Jazz V,Honda,Jazz V
3,Maruti Ertiga VDI,Maruti,Ertiga VDI
4,Audi A4 New 2.0 TDI Multitronic,Audi,A4 New 2.0 TDI Multitronic


In [4]:
df['Brand'].nunique(), df['Model'].nunique()

(31, 1876)

In [5]:
df['Year'].max()

np.int64(2019)

Using this dataset's max `Year` as the reference point, rather than today's
real-world date — that keeps `Car_Age` tied to when the listings were
actually collected, and reproducible no matter when this notebook is re-run.

In [6]:
REFERENCE_YEAR = df['Year'].max()
df = add_car_age(df, reference_year=REFERENCE_YEAR)
df[['Year', 'Car_Age']].head()

,Year,Car_Age
0,2010,9
1,2015,4
2,2011,8
3,2012,7
4,2013,6


In [7]:
df_check = engineer_features(
    pd.read_csv("../data/processed/used_cars_cleaned.csv"),
    reference_year=REFERENCE_YEAR
)
pd.testing.assert_frame_equal(
    df.reset_index(drop=True), df_check.reset_index(drop=True)
)
print("engineer_features() reproduces the step-by-step result exactly.")

engineer_features() reproduces the step-by-step result exactly.


In [8]:
df['Model'].value_counts().tail(20)

Model
Laura Classic 1.8 TSI                 1
Swift Dzire VXI Optional              1
Alto K10 LXI CNG                      1
Scorpio VLX 2WD Airbag BSIII          1
Figo Aspire 1.5 TDCi Titanium Plus    1
Jeep MM 540 DP                        1
Ertiga VXI AT Petrol                  1
Laura L and K AT                      1
Corolla Altis 1.8 G CVT               1
New C-Class C 200 AVANTGARDE          1
Rapid Ultima 1.6 TDI Elegance         1
GLA Class 200 Sport                   1
Indica LEI                            1
i20 2015-2017 Magna                   1
New Safari DICOR 2.2 VX 4x2           1
Elantra SX                            1
Wagon R Duo Lxi                       1
Polo IPL II 1.2 Petrol Highline       1
Bolt Revotron XT                      1
Xylo D4 BSIV                          1
Name: count, dtype: int64

A large share of `Model` values appear only a handful of times — confirms the
high-cardinality problem flagged in EDA. Decision on how to handle it
(bucket rare models into "Other", drop `Model` in favor of `Brand` alone,
target/frequency encoding) is made in the preprocessing pipeline, since it
needs a frequency count fit on the training split only.

In [9]:
df.to_csv("../data/processed/used_cars_features.csv", index=False)
print(f"Saved feature-engineered dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Saved feature-engineered dataset: 6018 rows, 17 columns
